In [ ]:
%pip install tabpfn
%pip install tabpfn-client
%pip install matplotlib
%matplotlib inline
%pip install rich

import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from IPython.display import clear_output
import matplotlib.pyplot as plt

# 🛠️ Backend-Auswahl: Lokale Ausführung vs. Cloud-API

TabPFN bietet zwei Betriebsmodi für flexible Einsatzszenarien:

1. **Lokaler Modus (Standard & Datenschutz-Fokus):**
   * **Vorteil**: Die Berechnungen und das Training finden vollständig auf deiner Hardware (oder der Colab-VM) statt. Deine Tabellendaten werden niemals an externe Server übertragen (vollständig datenschutzkonform für sensible Daten).
   * **Hinweis**: Erfordert einmalig einen kostenlosen Prior Labs Account und die Lizenzbestätigung beim ersten Download der Modellgewichte.
2. **Cloud-API Modus (`tabpfn_client`):**
   * **Vorteil:** Verlagert die Berechnung auf die Server von Prior Labs.
   * **Ideal für:** Schwächere Laptops oder schnelle Testumgebungen ohne dedizierte Hardware.

> **Hinweis zur Einrichtung:**
> Für beide Nutzungsverfahren ist ein kostenloser Account bei Prior Labs erforderlich. Führe das folgende Skript aus und befolge die Anweisungen

HIER: **Betriebsart auswählen**

In [ ]:
# True  = Über die Cloud-API (schont schwache Laptops, erfordert einen Prior Labs Token)
# False = Lokal auf dem Computer (vollständig datenschutzkonform, läuft komplett offline)
USE_CLOUD_API = True  

In [ ]:
import os
import getpass
from rich.console import Console
from rich.panel import Panel
from rich.rule import  Rule

console = Console()

# --- Automatische Authentifizierung (für lokal & client gleichermaßen nötig) ---
tabpfn_token = None
try:
    from google.colab import userdata
    tabpfn_token = userdata.get("TABPFN_TOKEN")
    if tabpfn_token:
      os.environ["TABPFN_TOKEN"] = tabpfn_token
except Exception:
  pass

if not tabpfn_token and not os.environ.get("TABPFN_TOKEN"):
  console.print(
      Panel(
          "So erhältst du deinen kostenlosen API-Key:\n\n"
          "  1. Gehe auf [link=https://ux.priorlabs.ai]ux.priorlabs.ai[/link] und registriere dich\n"
          "  2. Akzeptiere die Lizenz unter [link=https://ux.priorlabs.ai/account/licenses]ux.priorlabs.ai/account/licenses[/link]\n"
          "  3. Kopiere deinen API-Key von [link=https://ux.priorlabs.ai/account]ux.priorlabs.ai/account[/link]\n\n"
          "[bold yellow]Tipp:[/bold yellow] Trage den Key einmalig links in Colab unter dem "
          "[bold cyan]Schlüsselsymbol (Secrets)[/bold cyan] als [bold cyan]TABPFN_TOKEN[/bold cyan] ein.",
          title="[bold]🔑 Prior Labs API-Key erforderlich[/bold]",
          border_style="blue",
      )
  )
  while not tabpfn_token:
    token_input = getpass.getpass("Füge deinen TABPFN_TOKEN ein und drücke Enter: ")
    if token_input.strip():
      tabpfn_token = token_input.strip()
      os.environ["TABPFN_TOKEN"] = tabpfn_token
    else:
      console.print("[red]Der Token darf nicht leer sein. Bitte versuche es erneut.[/red]")

# --- Backend-Auswahl ---
if USE_CLOUD_API:
  try:
    from tabpfn_client import TabPFNClassifier, init, set_access_token

    set_access_token(os.environ.get("TABPFN_TOKEN"))
    init()
    classifier = TabPFNClassifier()
    console.print(
        "✅ [bold green]Modus: Cloud-API aktiv (Berechnung auf Prior Labs"
        " Servern).[/bold green]"
    )
  except ImportError:
    console.print(
        "❌ [bold red]Fehler: 'tabpfn_client' ist nicht installiert. Bitte"
        " führe '!pip install tabpfn-client' aus.[/bold red]"
    )
else:
  # Lokale Variante (Datenschutz-Fokus für KMUs)
  from tabpfn import TabPFNClassifier

  classifier = TabPFNClassifier(device='cpu', ignore_pretraining_limits=True)
  console.print("✅ [bold green]Modus: Lokale Ausführung aktiv[/bold green]")

# KLASSIFIKATION

## Datensatz laden und bereinigen


In [ ]:
# 1. Datensatz aus Github-URL als CSV einlesen
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
data = pd.read_csv(url)

# 3. Spaltennamen ins Deutsche übersetzen
spalten_mapping = {
    'gender': 'Geschlecht',
    'SeniorCitizen': 'Senior',
    'Partner': 'Partner',
    'Dependents': 'Kinder_im_Haushalt',
    'tenure': 'Vertragsdauer_Monate',
    'PhoneService': 'Telefon_Dienst',
    'MultipleLines': 'Mehrfachleitungen',
    'InternetService': 'Internetdienst',
    'OnlineSecurity': 'Online_Sicherheit',
    'OnlineBackup': 'Online_Backup',
    'DeviceProtection': 'Geräteschutz',
    'TechSupport': 'Technischer_Support',
    'StreamingTV': 'Streaming_TV',
    'StreamingMovies': 'Streaming_Filme',
    'Contract': 'Vertragsart',
    'PaperlessBilling': 'Papierlose_Rechnung',
    'PaymentMethod': 'Zahlungsmethode',
    'MonthlyCharges': 'Monatliche_Kosten_Euro',
    'TotalCharges': 'Gesamtkosten_Euro',
    'Churn': 'Kuendigung'
}
data = data.rename(columns=spalten_mapping)


data.head()


# 2.1 TotalCharges von Text in echte Zahlen umwandeln
data['Gesamtkosten_Euro'] = pd.to_numeric(data['Gesamtkosten_Euro'], errors='coerce').fillna(0)

# 2.2 customerID löschen (das ist nur ein Name, kein Merkmal fürs Modell)
data = data.drop(columns=['customerID'])

# 2.3 Zielvariable Churn (Yes/No) in 1 und 0 umwandeln
data['Kuendigung'] = data['Kuendigung'].map({'Yes': 1, 'No': 0})


# 3. Features und Zielvariable trennen
X = data.drop(columns=['Kuendigung'])
y = data['Kuendigung']


display(data)

## TabPFN-Modelltraining

In [ ]:
from tabpfn import TabPFNClassifier

# 1. Train-Test-Split (80% Training, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. TabPFN initialisieren (läuft lokal auf der CPU)
classifier = TabPFNClassifier(ignore_pretraining_limits=True)

print("Modell wird trainiert...")
classifier.fit(X_train, y_train)


# 3. Vorhersage und Genauigkeit berechnen
y_pred_tabpfn = classifier.predict(X_test)
tabpfn_risiko = classifier.predict_proba(X_test)[:, 1]

def bewerte_modell(name, y_test, y_pred, y_risiko):
    return {
        "Modell": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_risiko),
    }

tabpfn_results = bewerte_modell("TabPFN", y_test, y_pred_tabpfn, tabpfn_risiko)

clear_output(wait=True)

print("TabPFN Ergebnisse:", tabpfn_results)


## BENCHMARK: Vergleich zu Random Forest
**Warum Random Forest?** 
    Er gilt im Data Science als einer der robustesten und beliebtesten Standard-Klassifikatoren für tabellarische Daten. Er liefert solide Ergebnisse, ohne dass man stundenlang an Parametern schrauben muss – genau wie TabPFN. Das macht ihn zum idealen Vergleichspartner.


- **Die technische Hürde (Datenaufbereitung):** 
    Ein klassischer Random Forest kann rein mathematisch **nur mit Zahlen** rechnen. Textspalten wie `"Male"` / `"Female"` oder Verträge können von dem Algorithmus nicht direkt verarbeitet werden.
- **Die Lösung (Pipeline & ColumnTransformer):** 
    Mithilfe einer `Pipeline` und eines `ColumnTransformer` automatisieren wir diesen Schritt: Textspalten werden automatisch erkannt und per `OrdinalEncoder` in Zahlen übersetzt, während reine Zahlen-Spalten einfach unverändert durchgelassen werden. Das hält den Workflow modular und schützt vor Datenlecks.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier


# 1. Alle Spalten mit kategorischen Daten identifizieren
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Den ColumnTransformer einrichten
column_transformer = ColumnTransformer(
    transformers=[
        ('CatToNum', OrdinalEncoder(), categorical_cols)
    ],
    remainder='passthrough'  # Numerische Spalten einfach unverändert durchlassen
)

# 3. Pipeline erstellen: Erst umwandeln, dann Random Forest trainieren
rf_pipeline = make_pipeline(column_transformer, RandomForestClassifier(random_state=42))

print("Random Forest Pipeline wird trainiert...")
rf_pipeline.fit(X_train, y_train)

# 4. Vorhersage ausführen
y_pred_rf = rf_pipeline.predict(X_test)
rf_risiko = rf_pipeline.predict_proba(X_test)[:, 1]

rf_results = bewerte_modell("Random Forest", y_test, y_pred_rf, rf_risiko)

clear_output(wait=True) 

print("Random Forest Ergebnisse:", rf_results)

# Tabellarischer direkter Modell-Vergleich
vergleich_df = pd.DataFrame([tabpfn_results, rf_results])

display(vergleich_df.style.format({
    "Accuracy": "{:.3f}",
    "F1-Score": "{:.3f}",
    "ROC-AUC": "{:.3f}"
}).background_gradient(cmap='Blues', subset=['Accuracy', 'F1-Score', 'ROC-AUC']))

print("\n" + "="*60 + "\n")

# Visueller Vergleich: Side-by-Side Konfusionsmatrizen
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# A) TabPFN Konfusionsmatrix (Voraussetzung: y_pred_tabpfn existiert)
cm_tabpfn = confusion_matrix(y_test, y_pred_tabpfn)
disp_tabpfn = ConfusionMatrixDisplay(confusion_matrix=cm_tabpfn, display_labels=['Kein Churn', 'Churn'])
disp_tabpfn.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'TabPFN Classifier (Acc: {tabpfn_results["Accuracy"]:.3f})', fontsize=12, fontweight='bold')

# B) Random Forest Konfusionsmatrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['Kein Churn', 'Churn'])
disp_rf.plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title(f'Random Forest Classifier (Acc: {rf_results["Accuracy"]:.3f})', fontsize=12, fontweight='bold')

plt.suptitle('Direkter Modell-Vergleich: Konfusionsmatrizen (IBM Telco Churn)', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

- **Ergebnisse:** 
    Während der Random Forest ein stabiles, bewährtes Arbeitstier ist, zeigt der direkte Vergleich, wie sich moderne Ansätze (wie TabPFN) schlagen. Oft liefert TabPFN direkt out-of-the-box extrem starke Ergebnisse, ohne dass man sich um komplexe Hyperparameter-Suche (Grid Search) kümmern muss.
    
- **Fazit für die Praxis:** 
    Für KMUs ist oft nicht nur die reine Genauigkeit entscheidend, sondern auch die **Zeit** und der **Aufwand**, die man in die Datenaufbereitung und das Tuning stecken muss. Pipelines helfen uns dabei, diesen Prozess reproduzierbar und fehlerfrei zu halten.

## Welche Kunden sollte das Unternehmen zuerst kontaktieren?

Für die Risikoliste wird TabPFN verwendet, sofern es erfolgreich ausgeführt wurde.  
Andernfalls wird die Baseline genutzt.

In [ ]:
# Risikoliste zusammenbauen
risikoliste = X_test.copy()
risikoliste["tatsaechlich_kuendigung"] = y_test.values
risikoliste["vorhergesagte_kuendigung"] = y_pred_tabpfn
risikoliste["kuendigungsrisiko_prozent"] = (tabpfn_risiko * 100).round(1)

risikoliste = risikoliste.sort_values("kuendigungsrisiko_prozent", ascending=False)

# Die wichtigsten deutschen Spalten für die Top-10-Ansicht
anzeigespalten = [
    "kuendigungsrisiko_prozent", 
    "Vertragsart",         
    "Vertragsdauer_Monate",
    "Monatliche_Kosten_Euro",
    "Internetdienst",      
    "Zahlungsmethode"      
]

print("\n🎯 Top 10 gefährdete Kunden (Action-Item für den Vertrieb):")
display(risikoliste[anzeigespalten].head(10))


# Die Top 10 für das Diagramm vorbereiten und für die Darstellung sortieren
top10 = risikoliste.head(10).sort_values("kuendigungsrisiko_prozent")

plt.figure(figsize=(9, 5))
plt.barh(
    [f"Kunde #{idx}" for idx in top10.index],
    top10["kuendigungsrisiko_prozent"],
)
plt.title("🎯 Zehn höchste Kündigungsrisiken (Top 10 Action-Item)")
plt.xlabel("Vorhergesagtes Kündigungsrisiko in Prozent")
plt.ylabel("Kunden-ID (Tabellen-Index)")
plt.xlim(0, 100)
plt.grid(axis="x", alpha=0.25)

# Die Prozentwerte direkt an die Balken schreiben
for index, wert in enumerate(top10["kuendigungsrisiko_prozent"]):
    plt.text(wert + 1, index, f"{wert:.1f} %", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()



## Wo macht das Modell Fehler?

Die Konfusionsmatrix zeigt:

- korrekt erkannte bleibende Kunden  
- korrekt erkannte Kündigungen  
- Fehlalarme  
- übersehene Kündigungen  

Gerade für Unternehmen ist wichtig, welche Fehlerart teurer ist.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_tabpfn,
    display_labels=["bleibt", "kündigt"],
    values_format="d",
)

plt.title(f"Konfusionsmatrix – TabPFN")
plt.tight_layout()
plt.show()

## Schlussfolgerung für den Vortrag

**Was TabPFN vereinfacht**

- kein eigenes neuronales Netz entwickeln  
- wenig Modellkonfiguration  
- schnelle Anwendung auf kleine tabellarische Datensätze  
- vertraute Scikit-learn-Schnittstelle  

**Was trotzdem Aufgabe des Unternehmens bleibt**

- sinnvolle Geschäftsfrage auswählen  
- Datenqualität prüfen  
- ungeeignete Merkmale und Datenlecks vermeiden  
- Vorhersagen richtig bewerten  
- konkrete Maßnahmen aus zuverlässige KI ableiten  

> Eine hohe Kündigungswahrscheinlichkeit ist keine Gewissheit, sondern ein Hinweis darauf, wo sich ein genauerer Blick lohnen kann.

# REGRESSION

## Datensatz laden und bereinigen

In [ ]:
from tabpfn import TabPFNRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path
from sklearn.datasets import fetch_california_housing

california_housing = fetch_california_housing(as_frame=True)

housing_mapping = {
    "MedInc": "Mittleres_Einkommen",
    "HouseAge": "Hausalter_Jahre",
    "AveRooms": "Durchschnitt_Zimmer",
    "AveBedrms": "Durchschnitt_Schlafzimmer",
    "Population": "Bevoelkerung_Gebiet",
    "AveOccup": "Durchschnitt_Haushaltsgroesse",
    "Latitude": "Breitengrad",
    "Longitude": "Laengengrad",
    "MedHouseValue": "Immobilienpreis"
}

data2 = california_housing.data.rename(columns=housing_mapping)
data2['Immobilienpreis'] = california_housing.target

# Features und Zielvariable trennen
X_reg = data2.drop(columns=['Immobilienpreis'])
y_reg = data2['Immobilienpreis'] * 100000  # Umrechnung in echte Euro/Dollar-Werte

display(X_reg.head())


## TabPFN-Modelltraining

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# TabPFN Regressor initialisieren und trainieren
print("Trainiere TabPFN Regressor...")
regressor = TabPFNRegressor(ignore_pretraining_limits=True)
regressor.fit(X_train_r, y_train_r)

# Vorhersagen treffen
y_pred_reg = regressor.predict(X_test_r)

clear_output(wait=True)
print("Vorhersagen abgeschlossen.")

## Evaluation

In [ ]:
# Metriken berechnen
mae = mean_absolute_error(y_test_r, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_reg))
r2 = r2_score(y_test_r, y_pred_reg)

print(f"\n📊 Regressions-Auswertung für TabPFN:")
print(f"  • Mittlerer absoluter Fehler (MAE):  {mae:,.2f} €")
print(f"  • Wurzel des mittleren Fehlers (RMSE): {rmse:,.2f} €  (bestraft große Ausreißer stärker)")
print(f"  • Bestimmtheitsmaß (R² Score):       {r2:.3f}")

# Visueller Vergleich als Scatterplot
plt.figure(figsize=(7, 6))
plt.scatter(y_test_r, y_pred_reg, alpha=0.4, color="teal", edgecolors="none")

min_val = min(y_test_r.min(), y_pred_reg.min())
max_val = max(y_test_r.max(), y_pred_reg.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label="Perfekte Vorhersage")

plt.xlabel("Echter Immobilienpreis (€)")
plt.ylabel("Vorhergesagter Preis durch TabPFN (€)")
plt.title("TabPFN Regression: Echte vs. vorhergesagte Immobilienpreise")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## BENCHMARK: Vergleich zu Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# 1. Random Forest Regressor initialisieren und trainieren (als Baseline)
print("\n🌲 Trainiere Random Forest Regressor als Vergleichsmodell...")
rf_regressor = RandomForestRegressor(random_state=42, n_estimators=100)
rf_regressor.fit(X_train_r, y_train_r)

# 2. Vorhersagen für Random Forest treffen
y_pred_rf = rf_regressor.predict(X_test_r)

# 3. Metriken berechnen
mae_rf = mean_absolute_error(y_test_r, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_r, y_pred_rf))
r2_rf = r2_score(y_test_r, y_pred_rf)

print(f"\n📊 Regressions-Auswertung (Random Forest):")
print(f"  • Mittlerer absoluter Fehler (MAE):  {mae_rf:,.2f} €")
print(f"  • Wurzel des mittleren Fehlers (RMSE): {rmse_rf:,.2f} €")
print(f"  • Bestimmtheitsmaß (R² Score):       {r2_rf:.3f}")

# 4. Kurzer direkter Tabellen-Vergleich für den Workshop
print("\n" + "="*50)
print(" 🥊 MODELL-VERGLEICH: TabPFN vs. Random Forest ")
print("="*50)
print(f" {'Metrik':<20} | {'TabPFN':<15} | {'Random Forest':<15}")
print("-" * 55)
print(f" {'R² Score (höher=besser)':<20} | {r2:<15.3f} | {r2_rf:<15.3f}")
print(f" {'MAE (niedriger=besser)':<20} | {mae:,.2f} €   | {mae_rf:,.2f} €   ")
print(f" {'RMSE (niedriger=besser)':<20} | {rmse:,.2f} €   | {rmse_rf:,.2f} €   ")
print("="*50)


# Abbildung mit zwei nebeneinanderliegenden Plots erstellen
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# 1. TabPFN Scatterplot
axes[0].scatter(y_test_r, y_pred_reg, alpha=0.3, color='#1f77b4', edgecolor='none')
axes[0].plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 'r--', lw=2, label='Ideale Linie (y = x)')
axes[0].set_title(f'TabPFN Regressor (R² = {r2:.3f})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tatsächliche Preise', fontsize=10)
axes[0].set_ylabel('Vorhergesagte Preise', fontsize=10)
axes[0].legend(loc='upper left')
axes[0].grid(True, linestyle=':', alpha=0.6)

# 2. Random Forest Scatterplot
axes[1].scatter(y_test_r, y_pred_rf, alpha=0.3, color='#2ca02c', edgecolor='none')
axes[1].plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 'r--', lw=2, label='Ideale Linie (y = x)')
axes[1].set_title(f'Random Forest Regressor (R² = {r2_rf:.3f})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tatsächliche Preise', fontsize=10)
axes[1].legend(loc='upper left')
axes[1].grid(True, linestyle=':', alpha=0.6)

# Gesamttitel für das Diagramm
plt.suptitle('Direkter Modell-Vergleich: Tatsächliche vs. Vorhergesagte Immobilienpreise', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()